# Connect an Agent to Doris and Check Its Evidence

**Module 1: Connect AI Agents to Apache Doris with MCP** · [Lesson](course.md) · [Quiz](quiz01_doris_mcp.ipynb)

In this Lab you start the course's Doris sandbox, load a small table of storefront events, create the read-only account that the MCP Server uses, and run the MCP launcher the way an MCP Host runs it. Then you connect a Host and ask it three questions, following sections 1.5 and 1.6 of the [lesson](course.md). The second half of the Lab checks the Host's answers: it runs the decisive SQL directly, makes the same three tool calls without a model, and reads the audit trail that both the MCP Server and Doris keep.

Time: about 30 minutes, plus the time to connect your Host. The first run downloads the Doris image: about 1.9 GB, 4.5 GB once unpacked.

Before you start: Docker is running, and you opened this notebook from the course's virtual environment (see Get started in the [course README](../../README.md)). No other course is needed.

### Initialize the Lab

Run this cell once. It finds the course directory, loads the course's lab helpers, and shows the Doris address and the launcher path that the Lab uses.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

## 1. Start the Doris sandbox

The sandbox is one container, `doris-ai`, that runs one Frontend (FE) and one Backend (BE) of Apache Doris 4.1.3. Its three ports listen only on `127.0.0.1`:

| Port | Purpose | Variable to change it |
| --- | --- | --- |
| 39030 | FE MySQL protocol: SQL from this Lab and from the MCP Server | `DORIS_AI_QUERY_PORT` |
| 38030 | FE HTTP | `DORIS_AI_HTTP_PORT` |
| 38040 | BE HTTP: the memory metrics in step 3 of the lesson | `DORIS_AI_BE_HTTP_PORT` |

To use other ports, set the variables before you start Jupyter. If you use your own Doris 4.x instead of the sandbox (lesson section 1.3), skip this cell.

In [ ]:
lab.prepare_environment()

**Expected result**

The host port check lists the three ports as `available`, or as `used by this course` when you rerun the cell. The nodes table shows one FE and one BE, both alive, with version `doris-4.1.3-rc02-7126cf65d96`: that is how the 4.1.3 release reports itself. The last card gives the three addresses. The first run pulls the image and takes a few minutes; a rerun reuses the running container and takes a few seconds.

## 2. Load the storefront events

The course table `ai_course.events` holds 14,307 synthetic storefront events from March 2026: 800 shoppers in four regions view, add to the cart, and buy 200 products. The data comes from `datasets/events.csv` in the course directory; `datasets/generate_events.py` generated it, so it contains no real shoppers or orders.

The cell creates the database and the table, with a comment on the table and on every column, loads the CSV file, and compares the table with the numbers in `datasets/events.yaml`. It then waits until `information_schema.tables` reports the new row count. Doris refreshes that table statistic about once a minute, and the MCP Server's catalog tool reads its row count from it, not from `COUNT(*)`.

In [ ]:
lab.load_dataset()

**Expected result**

The comparison table shows the same value in both columns for every check: 14,307 rows, first event `2026-03-01 00:10:30`, last event `2026-03-31 23:06:38`, 800 distinct shoppers, total revenue `73236.22`, and 12,753 view, 1,074 cart, and 480 purchase events. On the first load, the statistics wait prints `TABLE_ROWS = 0` a few times and ends with a card that reports 14,307 rows; this takes up to two minutes. A rerun finds the data already loaded and finishes at once.

## 3. Create the read-only account and the launcher's environment file

The MCP Server connects to Doris as `mcp_reader`, an account that may only read the course database. The cell generates a random password, creates the account, and grants it `SELECT_PRIV` on `internal.ai_course.*`. It then writes `.env.mcp.local` next to the launcher, from `.env.mcp.local.example`: the Doris address, the account and its password, and the FE and BE HTTP addresses. Only your user can read the file (mode 600), and Git ignores it. When the file already exists, the cell keeps its password.

In [ ]:
lab.create_mcp_account()

**Expected result**

The statements show `••••••••` instead of the password. `SHOW GRANTS` lists `internal.ai_course: Select_priv`; Doris adds `Select_priv` on `internal.information_schema` and `internal.mysql` for every account, and `normal: Usage_priv` lets the account use the default workload group. The file view shows `.env.mcp.local` with mode `600` and the password hidden.

Before a Host uses the account, check what it can do. The next cell connects as `mcp_reader` and tries one read, one write, one table creation, and one read of the Doris audit log. The write inserts no rows even if it were allowed.

In [ ]:
lab.check_mcp_account()

**Expected result**

Only the read is allowed and returns `{"row_count": 14307}`. Doris denies the other three: `LOAD command denied to user 'mcp_reader'...`, `Access denied; you need (at least one of) the (CREATE) privilege(s) for this operation`, and `Permission denied ... [__internal_schema].[audit_log]`. The last denial matters later: several MCP children that read the audit log report themselves as not callable for this account.

## 4. Run the launcher the way a Host does

Every Host configuration in this module contains one thing: the absolute path of `run-doris-mcp-local`. The launcher reads `.env.mcp.local` and starts `doris-mcp-server --transport stdio` from the course's virtual environment. A Host starts it as a child process and talks to it over standard input and output.

The next cell does the same with the MCP Python SDK: it starts the launcher with `/` as the working directory and only a few inherited environment variables, as a desktop Host does, and asks for the tool list.

In [ ]:
lab.check_launcher();

**Expected result**

The tool list has eight rows, all marked as domains: `doris_catalog`, `doris_cluster`, `doris_governance`, `doris_lakehouse`, `doris_pipeline`, `doris_query`, `doris_search`, and `doris_semantic`. The card names the Server (`doris-mcp-server 1.0.0`) and the protocol version (`2025-11-25`, the handshake that desktop Hosts use). The Server wrote its logs to `logs/` next to the launcher; that directory is ignored by Git.

### Connect your Host now

The next cell shows the configuration of each Host with this computer's launcher path filled in. Use one of them and follow lesson section 1.5 to check the connection, then ask the three questions of section 1.6 in one conversation. Come back to section 5 of this Lab when the Host has answered.

In [ ]:
lab.host_configs()

**Expected result**

Four configuration blocks. Each contains the absolute path of `run-doris-mcp-local` on this computer and no password.

## 5. Check the agent's numbers

In step 2 of section 1.6, the agent writes SQL from the table context and runs it through the `execute_query` child. Run the decisive SQL yourself, as the same account, and compare every row with the agent's answer.

In [ ]:
WEEKLY_ACTIVITY_SQL = """
SELECT
    event_type,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_id) AS distinct_users
FROM ai_course.events
WHERE event_time >= '2026-03-09 00:00:00'
  AND event_time <  '2026-03-16 00:00:00'
GROUP BY event_type
ORDER BY event_count DESC
LIMIT 10
"""

lab.reader_query(WEEKLY_ACTIVITY_SQL);

**Expected result**

| `event_type` | `event_count` | `distinct_users` |
| --- | ---: | ---: |
| `view` | 2904 | 453 |
| `cart` | 238 | 204 |
| `purchase` | 106 | 94 |

If the agent's rows differ, compare its SQL with this one: the time bounds, the grouping, and the order are the usual suspects.

## 6. Make the three tool calls without a model

The agent chose its tools and arguments itself. The next cell makes the same calls through the same launcher, with fixed arguments, so you can compare the shape of your Host's trace with a known one. For each step it first calls the domain with `{}` (discovery), then calls the same domain with one `child_tool`, its `arguments`, and the `manifest_version` that discovery returned.

Each call shows a summary, the children that discovery reported, the important fields of the result, and the complete structured result in a collapsed block.

In [ ]:
trace = lab.reference_trace(WEEKLY_ACTIVITY_SQL)

**Expected result**

- **Step 1.** Discovery of `doris_catalog` lists five children, all callable. `get_table_context` returns the seven columns with their types and comments; `event_time` and `event_id` are the key columns, the `indexes` section reports 0 indexes, and every section has status `success` and no warnings. `basic.row_count` is 14307, the table statistic that section 2 waited for.
- **Step 2.** Discovery of `doris_query` lists seven children; `list_slow_queries` and the two ADBC children are not callable (`RUNTIME_PROBE_FAILED`: `mcp_reader` cannot read the audit log; `PROVIDER_NOT_CONFIGURED`: ADBC is off). `execute_query` returns the three rows of section 5, with operation `SELECT`, `truncated` false, no warnings, and a row ceiling (`max_rows`) of 100.
- **Step 3.** Discovery of `doris_cluster` lists eleven children, some of them not callable for this account or this deployment. `get_memory_stats` returns one BE node, `configured_0`, with status `success`, no warnings, source `doris_be_metrics`, and `invented_values` false. The byte values change on every run.

The `manifest_version` values (`catalog.…`, `query.…`, `cluster.…`) are opaque strings; yours may differ from your Host's.

## 7. Read the audit trail

Two logs record what happened, at different layers:

- The MCP Server's audit log, `logs/doris_mcp_server_audit.log` next to the launcher, has one line per tool call: the domain or child, the status, the duration, and the names of the arguments. It does not record argument values, so no SQL text and no table names.
- Doris records every statement that `mcp_reader` runs in its audit table, `__internal_schema.audit_log`, which only an administrator can read. The Lab reads it with the administrator account.

In [ ]:
lab.server_audit_log();

**Expected result**

The last entries are the six calls of section 6, in order: `doris_catalog`, `doris_catalog.get_table_context`, `doris_query`, `doris_query.execute_query`, `doris_cluster`, `doris_cluster.get_memory_stats`, each with status `success`. The child calls show their argument names, `database,table` and `sql`; `get_memory_stats` had none. If your Host made calls since then, they appear too.

In [ ]:
lab.doris_audit_log();

**Expected result**

Doris writes audit entries in batches, at most every 60 seconds, so the cell may wait about a minute. The table lists, in time order, the statements that name `ai_course`, each with state `EOF` (a completed query): the weekly `SELECT` that section 5 ran directly, the five metadata statements behind `get_table_context` (`SHOW FULL COLUMNS FROM ai_course.events`, two `information_schema` queries for the comments, `SHOW INDEX`, and the `information_schema.tables` query that returns `TABLE_ROWS`), and the weekly `SELECT` again, this time from `execute_query`. Doris cannot tell the two `SELECT` statements apart: both ran as `mcp_reader`. A card counts the other statements of the same account in that period, about 30: the Server's connection checks and capability probes, such as `SELECT 1` and `SHOW FRONTENDS`. Doris shows times in the sandbox's time zone, UTC; the Server's log uses this computer's time zone.

## 8. Exercise: break the route and find the cause

Lesson section 1.7 says to troubleshoot from Doris outward. Practice it on a failure you cause yourself:

1. Change one character of `DORIS_PASSWORD` in `.env.mcp.local`. Keep the correct value in a variable first (`lab.env_value("DORIS_PASSWORD")`), and change the file with `lab.update_env_file(DORIS_PASSWORD=...)`.
2. Run `lab.check_launcher()`. Which layer reports the failure, and what does `logs/doris_mcp_server_error.log` say about the cause?
3. Run `lab.check_reader_login()`. What does Doris itself say?
4. Put the correct password back and run `lab.check_launcher()` again.

A Host would show the same failure as a Server that failed to start or disconnected. Write your code in the empty cell; the reference solution follows in a collapsed cell.

In [ ]:
# Your code here

**Reference solution.** Expand the next cell after you try; it runs the four steps.

In [ ]:
correct = lab.env_value("DORIS_PASSWORD")
wrong = correct[:-1] + ("x" if correct[-1] != "x" else "y")

lab.update_env_file(DORIS_PASSWORD=wrong)
lab.check_launcher()        # the Server exits: its error log says the connectivity test failed
lab.check_reader_login()    # Doris names the cause: access denied for mcp_reader

lab.update_env_file(DORIS_PASSWORD=correct)
lab.check_launcher();       # the eight domain tools are back

**Expected result**

With the wrong password, the launcher check shows a failure card: the client reports `MCPError: Connection closed`, because the Server exited during startup, and the error log lines include `Database connectivity test failed` and `Failed to connect to Doris database within 30.0 seconds. Please check if Doris is running`, with the reason replaced by `[REDACTED]`. The Server cannot tell a wrong password from a stopped Doris in its log; the direct login can: `Access denied for user 'mcp_reader@...' (using password: YES)`. After the password is restored, the check lists the eight domain tools again. No Doris privilege had to change.

## 9. Completion criteria

- [ ] Section 2 reports 14,307 rows, and the table statistics are up to date.
- [ ] `mcp_reader` can read `ai_course` and is denied writes, DDL, and the audit log.
- [ ] `.env.mcp.local` exists with mode 600, and the launcher check lists eight domain tools.
- [ ] Your Host connected and answered the three questions of lesson section 1.6.
- [ ] The agent's rows match section 5.
- [ ] You compared your Host's tool calls with section 6 and found the agent's statements in the Doris audit table.
- [ ] You restored the password after the exercise, and the launcher check passes.

## 10. Stop or remove the environment

To free memory and keep everything for later, stop the sandbox; `lab.start()` starts it again with the same data. While it is stopped, a Host that starts the launcher reports the Server as failed.

In [ ]:
lab.stop()

In [ ]:
lab.start()

**Expected result**

`docker stop doris-ai` prints the container name and a card confirms the stop; the next cell starts the container again and waits until it is healthy.

When you have finished the module, remove the doris entry from your Host first (Clean up, in the lesson). Then set `REMOVE_EVERYTHING = True` and run the next cell: it deletes the `doris-ai` container, its two volumes and its network, `.env.mcp.local`, and `logs/`. The `mcp_reader` account goes with the sandbox. With your own Doris, run `lab.drop_mcp_account()` instead and drop the `ai_course` database yourself.

In [ ]:
REMOVE_EVERYTHING = False

if REMOVE_EVERYTHING:
    lab.teardown()
else:
    print("Nothing removed. Set REMOVE_EVERYTHING = True and run this cell again.")

**Expected result**

With `REMOVE_EVERYTHING = False`, the cell prints that nothing was removed. With `True`, a card lists the removed container, volumes, network, and files, and `docker ps -a --filter name=doris-ai` shows no containers.

## Next

Open [Quiz 1: Connect AI Agents to Apache Doris with MCP](quiz01_doris_mcp.ipynb). This is the only module of the course for now; the [course README](../../README.md#roadmap) lists the modules that are planned.